In [17]:
import datasets
from transformers import AutoModelForCausalLM, AutoTokenizer


PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [6]:
ds_dict = datasets.load_dataset("dongwonj/Execution-Grounded-Reasoning")


data/train-00000-of-00001.parquet:   0%|          | 0.00/45.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14743 [00:00<?, ? examples/s]

In [9]:
train = ds_dict['train']

In [10]:
train

Dataset({
    features: ['user_prompt', 'assistant_prompt', 'input', 'answer', 'execution_trace', 'refcode'],
    num_rows: 14743
})

In [11]:
pwd

'/lustre/hdd/LAS/weile-lab/mdrahman/MoECoder'

In [12]:
train.to_json("./data/execution_grounded_reasoning/all.jsonl")

Creating json from Arrow format:   0%|          | 0/15 [00:00<?, ?ba/s]

156360781

In [39]:
messages_list = []

for row in train:
    messages = [
        {"role": "user", "content": row['user_prompt']},
        {"role": "assistant", "content": row['assistant_prompt']},
    ]
    messages_list.append(messages)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B")
inputs = tokenizer.apply_chat_template(
    messages_list,
    tokenize=False,
    add_generation_prompt=False
) 
org_count = len(inputs)
messages_list = [data for i, data in enumerate(messages_list) if len(inputs[i]) <= 8192]

In [40]:
print(inputs[0])

<|im_start|>user
You are given a question that requires some input and output variables as follows:

Given a number `n`, where `n` represents the range of digits from 1 to `n`, find the sum of all unique products whose multiplicand/multiplier/product identity can be written as a 1 through `n` pandigital. What is the sum of these unique products?
Input:
  `n` (int): The number of digits to consider for the pandigital number. For example, if `n` is 5, the function will consider numbers from 1 to 5.
Output:
  `return` (int): The sum of all unique products whose multiplicand/multiplier/product identity can be written as a 1 through `n` pandigital.

----

Here is the solution code that solves the question:

```
# import necessary packages
from itertools import permutations
from typing import List

# all class and function definitions in the code file, if any
def list_to_int(values: List) -> int:
    return int("".join(map(str, values)))

# main function
def main_solution(n: int) -> int:
   

In [37]:
messages_list[0]

[{'role': 'user',
  'content': 'You are given a question that requires some input and output variables as follows:\n\nGiven a number `n`, where `n` represents the range of digits from 1 to `n`, find the sum of all unique products whose multiplicand/multiplier/product identity can be written as a 1 through `n` pandigital. What is the sum of these unique products?\nInput:\n  `n` (int): The number of digits to consider for the pandigital number. For example, if `n` is 5, the function will consider numbers from 1 to 5.\nOutput:\n  `return` (int): The sum of all unique products whose multiplicand/multiplier/product identity can be written as a 1 through `n` pandigital.\n\n----\n\nHere is the solution code that solves the question:\n\n```\n# import necessary packages\nfrom itertools import permutations\nfrom typing import List\n\n# all class and function definitions in the code file, if any\ndef list_to_int(values: List) -> int:\n    return int("".join(map(str, values)))\n\n# main function\n

In [33]:
len(messages_list)

14743

In [34]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B")
inputs = tokenizer.apply_chat_template(
    messages_list,
    tokenize=False,
    add_generation_prompt=True
) 
org_count = len(inputs)
messages_list = [data for i, data in enumerate(messages_list) if len(inputs[i]) <= 8192]

In [35]:
len(messages_list)

14743

In [ ]:
o
    for data in total_data:
        question = (data['problem_description']+'\n'+data['io_requirements']).replace("\n\n", "\n")
        prompt = get_training_prompt(question=question, code=data['refcode'], input=data['input'])
        if '</think>' not in data['nl_trace']: # filter out data where thinking is not finished
            continue
        nl_trace = data['nl_trace'].split('</think>')[1].strip()
        messages = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": nl_trace},
        ]
        messages_list.append(messages)
        
    tokenizer = AutoTokenizer.from_pretrained(args.trained_model)
    inputs = tokenizer.apply_chat_template(
        messages_list,
        tokenize=False,
        add_generation_prompt=True
    ) 
    org_count = len(inputs)
    messages_list = [data for i, data in enumerate(messages_list) if len(inputs[i]) <= 8192]